In [ ]:
from pathlib import Path
import pandas as pd
import json
import interpro_proto_go_term_comparison as ipgtc

with open("../data/test_interpro.json", "r") as f:
    interpro = json.load(f)
interpro2go = ipgtc.load_interpro2go("../data/interpro2go.txt")
mf_go_ids = ipgtc.load_molecular_function_go_ids("../data/go-basic-2020-06-01.obo")

BASE_RESULTS = Path(".../where/your/results/are")  # <-- CHANGE THIS TO YOUR RESULTS DIRECTORY
METRICS_DIR = BASE_RESULTS / "metrics_only_MF_recalculated"
METRICS_DIR.mkdir(exist_ok=True)


In [2]:
all_rows = []

for model_dir in (BASE_RESULTS / "prototypes").iterdir():
    if not model_dir.is_dir():
        continue

    model_name = model_dir.name

    for k_dir in model_dir.iterdir():
        if not k_dir.is_dir() or not k_dir.name.startswith("K"):
            continue

        k = int(k_dir.name[1:]) 

        if k != 1024 or model_name not in ["esm_kmeans", "mincut_K64", "puffin_K64_v4"]:
            continue

        print(f"Processing model={model_name}, K={k}")

        # --------------------
        # Load inputs
        # --------------------
        try:
            prototype_assignments = pd.read_csv(
                k_dir / "assignments_test.csv"
            )
            segment_path_name = model_name
            segment_assignments = pd.read_csv(
                BASE_RESULTS / "segments" / segment_path_name / "test" / "test_residue_assignments.csv"
            )

        except:
            print(f"One or more files missing for model={model_name}, K={k}. Skipping.")
            continue
        enrichment_df = pd.read_csv(
                k_dir / "eval"  / "go_enrichment_train_all.csv"
            )
        # --------------------
        # Run evaluation
        # --------------------
        summary = ipgtc.run_evaluation_over_all_pdbs(
            interpro,
            segment_assignments,
            prototype_assignments,
            enrichment_df,
            interpro2go,
            mf_go_ids
        )

        # --------------------
        # Dump JSON (mirrored hierarchy)
        # --------------------
        out_dir = METRICS_DIR / model_name / f"K{k}"
        out_dir.mkdir(parents=True, exist_ok=True)

        with open(out_dir / "test_metrics.json", "w") as f:
            json.dump(summary, f, indent=2)
        # --------------------
        # Flatten for table
        # --------------------
        mean_block = summary["mean_over_pdbs"]

        # overall metrics
        for metric, value in mean_block["overall"].items():
            all_rows.append({
                "model": model_name,
                "K": k,
                "interpro_type": "overall",
                "metric": metric,
                "value": value,
            })

        # per InterPro type
        for ipr_type, metrics in mean_block.get("by_interpro_type", {}).items():
            for metric, value in metrics.items():
                all_rows.append({
                    "model": model_name,
                    "K": k,
                    "interpro_type": ipr_type,
                    "metric": metric,
                    "value": value,
                })


Processing model=esm_kmeans, K=1024
Processing model=mincut_K64, K=1024
Processing model=puffin_K64_v4, K=1024


In [3]:
metrics_df = pd.DataFrame(all_rows)
metrics_df.to_excel(METRICS_DIR / "test_aggregated_metrics_long.xlsx", index=False)

wide_df = metrics_df.pivot_table(
    index=["model", "K", "interpro_type"],
    columns="metric",
    values="value"
).reset_index()

wide_df.to_excel(METRICS_DIR / "test_aggregated_metrics_wide.xlsx", index=False)